# Norm Intermezzo: Context of magnitude/length/norm of a vector explained.

The norm (or magnitude or length) of a vector describe the same thing: the distance between the origin ('0') and the coordinate of the vector.


- What properties does the norm of a vector describe?
- What pattern is there to be found in number of words and its vector length?

### Settings

In [ ]:
# File paths
embedding_file = "embedding_corpus_free_3600_250606.pickle"
raw_corpus_file = "corpus_free_3600_250606.csv"

## Initialization

### Imports

In [ ]:
import numpy as np
import pandas as pd
import pickle

import matplotlib.pyplot as plt

### Functions

In [ ]:
# function of description above
def get_norm(matrix):
    """Calculate the norm of a vector"""
    norm = np.sqrt(
        np.sum(
            np.power(
                matrix, 
                2
            ),
        axis=1
        )
    )

    return norm

# function to divide vector by norm
# (..probably there is a numpy function for it.. but i failed to find it)
def normalize(matrix, scales):
    """Divide elements in matrix by given scale"""
    return np.array([vector / scales[idx] for idx, vector in enumerate(matrix)])

In [ ]:
def to_series(M):
    return pd.Series(M)

In [ ]:
def get_n_chars(df, idx):
    return df.loc[idx, "sentence_text"].str.len()

def get_n_words(df, idx):
    return df.loc[idx, "sentence_text"].str.split(' ').map(len)

### Load files

In [ ]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')

# Load sentence embeddings
with open(f"../../data/vectors/{embedding_file}", 'rb') as handle:
    embeddings = pickle.load(handle)

## What properties does the norm of a vector describe?

Plot distribution of vector norms and show sentences that are in the upper and lower part of the distribution.

In [ ]:
# Get the norm of the vectors
M_norm = get_norm(embeddings)

In [ ]:
# Plot distribution of vector norms
plt.hist(
    M_norm,
    bins= np.arange(10,16,0.25)
)

plt.title("Distribution of vector norms")
plt.xlabel("Vector norm")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# Describe
pd.Series(M_norm).describe()

In [ ]:
idx_mean_norm = np.where((13.025 < M_norm) & (M_norm < 13.075))[0]
idx_mean_norm[:25]

In [ ]:
# Get indexes of vectors where norm is:
# larger than 15
idx_high_norm = np.where(M_norm > 15.25)[0]

# smaller than 11.25
idx_low_norm = np.where(M_norm < 11.25)[0]

# near mean (and only take 24 sentences)
idx_mean_norm = np.where((13.025 < M_norm) & (M_norm < 13.075))[0][:25]

print(f"idx_high_norm number of sentences: {idx_high_norm.shape[0]}")
print(f"idx_mean_norm number of sentences: {idx_mean_norm.shape[0]}")
print(f"idx_low_norm number of sentences: {idx_low_norm.shape[0]}")

#### Large norm

Vectors with a large norm in our corpus are very short sentences (2.3 words long, where the largest is 6 words long). (Is this what we expect?)

In [ ]:
# Describe vectors
print("Describe selected long vectors")
L_d = to_series(M_norm[idx_high_norm])
L_d.describe()

In [ ]:
print("\n~~~\n".join(df_corpus.loc[idx_high_norm, "sentence_text"].values))

In [ ]:
# Described number of characters per sentence
print("Described characters")
L_chars = get_n_chars(df_corpus, idx_high_norm)
L_chars.describe()

In [ ]:
# Described number of words per sentence
print("Described words")
L_words = get_n_words(df_corpus, idx_high_norm)
L_words.describe()

#### Average norm

In [ ]:
# Describe vectors
print("Describe selected average vectors")
A_d = to_series(M_norm[idx_mean_norm])
A_d.describe()

In [ ]:
print("\n~~~\n".join(df_corpus.loc[idx_mean_norm, "sentence_text"].values))

In [ ]:
# Described number of characters per sentence
print("Described characters")
A_chars = get_n_chars(df_corpus, idx_mean_norm)
A_chars.describe()

In [ ]:
# Described number of words per sentence
print("Described words")
A_words = get_n_words(df_corpus, idx_mean_norm)
A_words.describe()

#### Small norm

In [ ]:
# Describe vectors
print("Describe selected short vectors")
S_d = to_series(M_norm[idx_low_norm])
S_d.describe()

In [ ]:
print("\n~~~\n".join(df_corpus.loc[idx_low_norm, "sentence_text"].values))

In [ ]:
# Described number of characters per sentence
print("Described characters")
S_chars = get_n_chars(df_corpus,idx_low_norm)
S_chars.describe()

In [ ]:
# Described number of words per sentence
print("Described words")
S_words = get_n_words(df_corpus,idx_low_norm)
S_words.describe()

### Summary of described sentences

In [ ]:
plt.boxplot(
    [L_d, A_d, S_d]
)

plt.title("Vector norms")
# plt.xlabel("Norm")
plt.ylabel("Norm")

plt.show()

In [ ]:
plt.boxplot(
    [L_chars, A_chars, S_chars]
)

plt.title("Number of characters")
# plt.xlabel("Norm")
plt.ylabel("Character frequency")

plt.show()

In [ ]:
# Overall distribution
O_words = df_corpus['sentence_text'].str.split(' ').map(len)
plt.hlines([O_words.mean()], 
           xmin= 0, xmax=4, 
           color='grey', alpha=0.5, label="Overall mean")
plt.axhspan(
    ymax = O_words.mean() + O_words.std()*2,
    ymin = O_words.mean() - O_words.std()*2,
    color = 'grey',
    alpha= 0.1,
    label= "Overall 95%"
)
plt.axhspan(
    ymax = O_words.mean() + O_words.std(),
    ymin = O_words.mean() - O_words.std(),
    color = 'grey',
    alpha= 0.1,
    label= "Overall 68%"
)

# Plot boxplot subsets
plt.boxplot(
    [L_words, A_words, S_words]
)


plt.title("Number of words")
plt.legend()
plt.ylabel("Word frequency")

plt.show()

## What pattern is there to be found in number of words and its vector length?

In [ ]:
# Add column with number of words
df_corpus["n_words"] = df_corpus["sentence_text"].str.split(' ').map(len)

In [ ]:
df_corpus['vector_norm'] = M_norm

In [ ]:
selection = df_corpus[['n_words', 'vector_norm']]#.sort_values("vector_norm")
selection

In [ ]:
plt.scatter(
    # selection['vector_norm'],
    selection["n_words"],
    selection['vector_norm'],
    alpha=0.05
)

plt.xlabel("Word count")
plt.ylabel("Vector norm")
plt.title("Vector norm vs word count per sentence")

plt.show()